In [34]:
import numpy as np
import scipy.signal as sps
import scipy.stats as stats

#from tutorial https://kaavyamaha12.medium.com/extracting-audio-features-using-librosa-3be4ff1fe57f
def get_features(x, fs):
    try:
        feats = {}
        feats['mean'] = np.mean(x)
        feats['std'] = np.std(x)
        feats['rms'] = np.sqrt(np.mean(x**2))
        feats['mav'] = np.mean(np.abs(x))
        feats['iEMG'] = np.sum(np.abs(x))
        feats['wl'] = np.sum(np.abs(np.diff(x)))
        feats['zcr'] = ((x[:-1]*x[1:]<0).sum()) 
        freqs, psd = sps.welch(x, fs=fs, nperseg=min(256, len(x)))
        total_power = np.sum(psd) + 1e-12
        feats['spec_entropy'] = -np.sum((psd/total_power) * np.log(psd/total_power + 1e-12))
        # median freq
        cumsum = np.cumsum(psd)
        median_freq = freqs[np.searchsorted(cumsum, total_power/2.0)]
        feats['median_freq'] = median_freq
        # skew/kurtosis
        feats['skew'] = stats.skew(x)
        feats['kurtosis'] = stats.kurtosis(x)
        return feats
    except Exception as e:
        print(f"Error during feature extraction {e}")
        return None
    

In [ ]:
import os
import pandas as pd
import librosa

def check_labels(folder: str):
    f = "." 
    normal_folder = os.path.join(f, "Normal")
    spont_folder = os.path.join(f, "Spontanaktivität")
    for file in os.listdir(folder):
        if file.endswith(".csv"):
            csv_path = os.path.join(folder, file)
            wav_file = os.path.splitext(file)[0] + ".wav"

            # check Normal folder
            normal_path = os.path.join(normal_folder, wav_file)
            spont_path = os.path.join(spont_folder, wav_file)

            if os.path.exists(normal_path):
                target_path = normal_path
            elif os.path.exists(spont_path):
                target_path = spont_path
            else:
                print(f"WAV file not found in either folder: {wav_file}")
                continue

            signal, fs= librosa.load(target_path)
            
            df = pd.read_csv(csv_path)
        
            features = []
            for index, row in df.iterrows():
                start_sample = int(row["Start Sample"])
                end_sample = int(row["End Sample"])
                segment = signal[start_sample:end_sample]
                segment_features = get_features(segment, fs)
                segment_features["file"] = target_path
                features.append(segment_features)

            features_df = pd.DataFrame(features)
            combined_df = pd.concat([df, features_df], axis=1)
            combined_df.to_csv("features.csv", index=False)


            
check_labels("./second_labeling")